# A Revolução da Visão Computacional

Visão computacional usando a biblioteca YOLO da Ultralytics.

O Ultralytics YOLO26 é um _framework_ de Inteligência Artificial versátil que suporta múltiplas tarefas de visão computacional.
O _framework_ pode ser utilizado para realizar detecção, segmentação, caixas delimitadoras orientadas (OBB), classificação e estimativa de pose.
Cada uma dessas tarefas possui um objetivo e um caso de uso diferentes, permitindo que você resolva diversos desafios de visão computacional com um único _framework_.

**Não há mágica, há matemática!**

In [2]:
import sys
if "google.colab" not in sys.modules:
    raise RuntimeError("Este notebook deve ser executado no Google Colab.")


In [3]:
# instalar a biblioteca
!pip install -q -U ultralytics

# fazer o upload de uma imagem sugerida ou uma imagem sua
#!gdown "https://drive.google.com/u/1/link_da_imagem.jpeg"
#!wget "https://link_da_imagem.jpeg"

In [4]:
# carregar o modelo YOLOv26 pré-treinado e realizar a detecção de objetos na imagem carregada
#!yolo task=detect mode=predict model=yolo26n.pt conf=0.5 source='/content/imagem.jpeg'

# o resultado da detecção de objetos será salvo no diretório "/runs/detect/predictX"

In [5]:
# Seguindo os docs do YOLO
from ultralytics import YOLO

MODEL_PT = "yolo26n.pt"
EPOCHS = 10
IMGSZ = 640
BATCH = 16
FRACTION = 0.5  # usar apenas metade do dataset em cada epoch (treino mais rapido)
VIDEO_URL = "https://www.youtube.com/watch?v=2cEJ5R6T_Dg"
NDJSON = "objects-phone-v3.ndjson"  # arquivo enviado ao colab

# EMBED CODE
# <iframe src="https://platform.ultralytics.com/embed/edwin-campos-2/datasets/objects-phone-v3" width="480" height="320" frameborder="0" style="border-radius:12px"></iframe>

# LINK (link do dataset no Ultralytics Platform)
LINK = "https://platform.ultralytics.com/edwin-campos-2/datasets/objects-phone-v3"

In [6]:
# Montar o Google Drive (robusto ao erro "credentials-propagation 404" do Colab).
DRIVE = "/content/drive"

import os
from google.colab import drive

def drive_ja_montado():
    return os.path.isdir(os.path.join(DRIVE, "MyDrive"))

if drive_ja_montado():
    print("Google Drive ja montado em", DRIVE)
else:
    try:
        drive.mount(DRIVE)
    except Exception as e:
        print("drive.mount() falhou:", type(e).__name__, "-", e)
        print("Tentando o fluxo classico (_mount) ...")
        try:
            drive._mount(DRIVE)
        except Exception as e2:
            print("_mount() tambem falhou:", type(e2).__name__, "-", e2)
            print()
            print("O erro 404 em /credentials-propagation e um problema da sessao do Colab.")
            print("Tente, nesta ordem:")
            print("  1. Runtime > Restart session e execute o notebook novamente.")
            print("  2. No painel de arquivos (icone de pasta) da esquerda, clique em")
            print("     Mount Drive - usa o fluxo do navegador e costuma funcionar.")
            print("  3. Ou use a celula FALLBACK abaixo (envio via API REST do Drive).")
            raise

if not drive_ja_montado():
    raise RuntimeError("Google Drive nao montado. Veja as dicas acima ou use a celula FALLBACK.")


Google Drive ja montado em /content/drive


In [7]:
# Baixar o NDJSON mais novo do Ultralytics Platform via API (check_file).
# A API key NAO fica salva no notebook:
#   1) le da secret 'ULTRALYTICS_API_KEY' do Colab (icone de cadeado a esquerda);
#   2) se nao existir, pede no prompt (getpass) - use o prefixo 'ul_'.
import os
from getpass import getpass
from google.colab import userdata

key = os.environ.get("ULTRALYTICS_API_KEY")
if not key:
    try:
        key = userdata.get("ULTRALYTICS_API_KEY")
    except Exception:
        key = None
if not key:
        key = getpass("Coloque sua API key do Ultralytics Platform: ")

os.environ["ULTRALYTICS_API_KEY"] = key

from ultralytics.utils.checks import check_file  # check_file mora em utils.checks

DATASET_URI = "ul://baguette/datasets/objects-phone-v3"  # username baguette
NDJSON = check_file(DATASET_URI, download_dir="/content/ndjson")
print("NDJSON mais novo baixado ->", NDJSON)
print("Tamanho:", round(os.path.getsize(NDJSON) / 1e6, 2), "MB")


NDJSON mais novo baixado -> /content/ndjson/baguette/datasets/objects-phone-v3/objects-phone-v3.ndjson
Tamanho: 10.42 MB


In [8]:
# Copiar o NDJSON para uma pasta do Drive (so roda se o Drive estiver montado).
import shutil

DESTINO_DRIVE = "/content/drive/MyDrive/LIA/objects-phone-v3.ndjson"
os.makedirs(os.path.dirname(DESTINO_DRIVE), exist_ok=True)
shutil.copy(NDJSON, DESTINO_DRIVE)

if os.path.getsize(DESTINO_DRIVE) == os.path.getsize(NDJSON):
    print("NDJSON copiado para o Drive ->", DESTINO_DRIVE)
else:
    raise RuntimeError("Copia incompleta para o Drive.")


NDJSON copiado para o Drive -> /content/drive/MyDrive/LIA/objects-phone-v3.ndjson


In [9]:
# Converter o NDJSON para o formato YOLO.
# A função baixa todas as imagens das URLs assinadas, organiza em images/{train,val,test}
# com labels/ correspondentes (um .txt por imagem) e gera o arquivo data.yaml.
# Se o dataset YOLO ja existir (images/ + labels/ + data.yaml), pula a conversao.
from pathlib import Path
CONVERTED_DIR = "/content/converted_datasets"
yaml_path = Path(CONVERTED_DIR) / "data.yaml"
if yaml_path.exists() and (Path(CONVERTED_DIR) / "images" / "train").exists():
    print("Dataset YOLO ja convertido. Pulando conversao ->", yaml_path)
else:
    # Jupyter/Colab ja roda um event loop: aplicar nest_asyncio permite usar asyncio.run().
    import asyncio
    import subprocess, sys
    try:
        import nest_asyncio
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nest-asyncio"])
        import nest_asyncio
    nest_asyncio.apply()

    from ultralytics.data.converter import convert_ndjson_to_yolo

    yaml_path = asyncio.run(convert_ndjson_to_yolo(NDJSON, output_path=CONVERTED_DIR))
    print("\ndata.yaml gerado ->", yaml_path)



data.yaml gerado -> /content/converted_datasets/objects-phone-v3-0307fefc/data.yaml


In [10]:
# Conferir o conteúdo do data.yaml e a contagem de imagens por split.
from ultralytics.utils import YAML
import yaml as _yaml

config = YAML.load(yaml_path)
print("Conteúdo do data.yaml:")
print(_yaml.safe_dump(config, default_flow_style=False, sort_keys=False))

for split in ("train", "val", "test"):
    n = len(list(yaml_path.parent.glob(f"images/{split}/*")))
    print(f"{split}: {n} imagens")


Conteúdo do data.yaml:
task: detect
name: objects phone v3
description: Street-level surveillance video frames capturing people using cellular
  phones in urban environments, annotated with bounding boxes for object detection
  tasks.
bytes: 3345261051
url: https://platform.ultralytics.com/baguette/datasets/objects-phone-v3
version: latest
created_at: '2026-09-02T05:03:58.215Z'
updated_at: '2026-09-02T05:03:58.215Z'
names:
  0: celular
test: images/test
train: images/train
val: images/val
hash: 0307fefc
complete: true

train: 31911 imagens
val: 1877 imagens
test: 1257 imagens


In [11]:
# Treinar com transfer learning OU pular para o best.pt ja existente.
# Pergunta no prompt: digite "s" para re-treinar; qualquer outra coisa usa o arquivo existente.
# Se optar por re-treinar, as celulas de validacao e teste (proximas) rodam no modelo novo.
import glob

from ultralytics import YOLO

best_pts = sorted(glob.glob("/content/runs/detect/*/weights/best.pt"))
resposta = input("Re-treinar o modelo? (s/N): ").strip().lower()
RETREINAR = resposta in ("s", "sim", "y", "yes")

if RETREINAR:
    model = YOLO(MODEL_PT)  # modelo base pre-treinado (ex.: yolo26n.pt)
    results = model.train(
        data=yaml_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        fraction=FRACTION,  # usa apenas metade do dataset por epoch
    )
    best_path = str(model.trainer.best)
    print("Re-treino concluido. Melhores pesos em:", best_path)
elif best_pts:
    best_path = best_pts[-1]
    print("Usando best.pt existente (sem re-treino) ->", best_path)
    print("Validacao e teste abaixo rodarao sobre esse arquivo.")
else:
    print("Nenhum best.pt encontrado; treinando do zero.")
    model = YOLO(MODEL_PT)
    results = model.train(
        data=yaml_path,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        fraction=FRACTION,
    )
    best_path = str(model.trainer.best)
    print("Treino concluido. Melhores pesos em:", best_path)


Usando best.pt existente (sem re-treino) -> /content/runs/detect/train/weights/best.pt
Validacao e teste abaixo rodarao sobre esse arquivo.


In [12]:
# Validar com os melhores pesos do treino no split de validacao.
import glob

best_path = sorted(glob.glob("/content/runs/detect/*/weights/best.pt"))[-1]
print("Usando pesos ->", best_path)

val_model = YOLO(best_path)
metrics = val_model.val(data=yaml_path, split="val", imgsz=IMGSZ, batch=BATCH)
print("Boas metricas, box P/mAP50-95 ->", metrics.box.map)


Usando pesos -> /content/runs/detect/train/weights/best.pt
Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1048.8±384.0 MB/s, size: 43.5 KB)
val: Scanning /content/converted_datasets/objects-phone-v3-0307fefc/labels/val.cache... 1877 images, 273 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1877/1877 246.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 118/118 3.1it/s 37.8s0.3s
                   all       1877       1649      0.869      0.801      0.888      0.601
Speed: 2.4ms preprocess, 6.3ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to /content/runs/detect/val-21
Boas metricas, box P/mAP50-95 -> 0.600985896335078


In [13]:
# Testar no split de teste (dados nunca vistos no treino/validacao).
test_model = YOLO(best_path)
metrics_test = test_model.val(data=yaml_path, split="test", imgsz=IMGSZ, batch=BATCH)
print("Teste, box P/mAP50-95 ->", metrics_test.box.map)


Ultralytics 8.4.138 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 836.2±467.0 MB/s, size: 82.2 KB)
val: Scanning /content/converted_datasets/objects-phone-v3-0307fefc/labels/test.cache... 1257 images, 123 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1257/1257 351.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 79/79 3.0it/s 26.1s0.3ss
                   all       1257       1148      0.847      0.835      0.891      0.592
Speed: 2.1ms preprocess, 5.1ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /content/runs/detect/val-22
Teste, box P/mAP50-95 -> 0.5918602714791206


In [14]:
# Baixar video do YouTube e salvar o original no Google Drive.
import subprocess, sys, shutil, glob

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'yt-dlp'])
print("Baixando video ->", VIDEO_URL)
subprocess.run([sys.executable, '-m', 'yt_dlp',
    "-o", "/content/video_input.%(ext)s",
    VIDEO_URL])

video_input = glob.glob("/content/video_input.*")[-1]
print("Video baixado ->", video_input)

# DRIVE ja montado na celula 5; salvar em MyDrive/LIA/
VIDEO_DRIVE = DRIVE + "/MyDrive/LIA/" + os.path.basename(video_input)
shutil.copy(video_input, VIDEO_DRIVE)
print("Original salvo no Drive ->", VIDEO_DRIVE)


Baixando video -> https://www.youtube.com/watch?v=2cEJ5R6T_Dg
Video baixado -> /content/video_input.mp4
Original salvo no Drive -> /content/drive/MyDrive/LIA/video_input.mp4


In [15]:
# Converter webm para mp4 e rodar yolo predict com diagnostico completo.
import subprocess, shutil, os

# converter
video_mp4 = "/content/video_input.mp4"
video_tmp = "/content/video_converted.mp4"
subprocess.run(["ffmpeg", "-y", "-i", video_input, "-c:v", "libx264", "-crf", "23", video_tmp], check=True, capture_output=True)
shutil.move(video_tmp, video_mp4)
print("Convertido:", video_mp4, f"({os.path.getsize(video_mp4) / 1e6:.1f} MB)")

# rodar predict e capturar saida
cmd = [
    "yolo", "predict",
    f"model={best_path}",
    f"source={video_mp4}",
    "conf=0.15",
    f"imgsz={IMGSZ}",
    "save=True",
    "exist_ok=True",
    "name=predict_video",
    "project=/content/runs/detect",
]
print("Rodando:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print("STDOUT:", result.stdout[-3000:] if result.stdout else "(vazio)")
print("STDERR:", result.stderr[-3000:] if result.stderr else "(vazio)")
print("Return code:", result.returncode)

# listar tudo em runs/detect/
for root, dirs, files in os.walk("/content/runs/detect"):
    for f in files:
        path = os.path.join(root, f)
        print(f"  {path}  ({os.path.getsize(path) / 1e6:.1f} MB)")
if not os.listdir("/content/runs/detect"):
    print("  (runs/detect vazio)")


CalledProcessError: Command '['ffmpeg', '-y', '-i', '/content/video_input.mp4', '-c:v', 'libx264', '-crf', '23', '/content/video_input.mp4']' returned non-zero exit status 1.

In [ ]:
# Salvar no Google Drive: conversao (data.yaml), treino, validacao e predicao do video.
import shutil

RESULTADOS = "/content/drive/MyDrive/LIA/objects-phone-v3"
os.makedirs(RESULTADOS, exist_ok=True)

def copiar_para_drive(origem, destino):
    if not os.path.exists(origem):
        print("Nao encontrado, pulando:", origem)
        return
    if os.path.exists(destino):
        shutil.rmtree(destino)
    shutil.copytree(origem, destino)
    print("Salvo:", destino)

# 1) conversao: data.yaml (a estrutura images/ + labels/ ja esta em /content/converted_datasets)
shutil.copy(yaml_path, os.path.join(RESULTADOS, "data.yaml"))
print("data.yaml salvo.")

# 2) treino
for d in sorted(glob.glob("/content/runs/detect/train*"))[-1:]:
    copiar_para_drive(d, os.path.join(RESULTADOS, os.path.basename(d)))

# 3) validacao
for d in sorted(glob.glob("/content/runs/detect/val*"))[-1:]:
    copiar_para_drive(d, os.path.join(RESULTADOS, os.path.basename(d)))

# 4) predicao do video
for d in sorted(glob.glob("/content/runs/detect/predict_video*"))[-1:]:
    copiar_para_drive(d, os.path.join(RESULTADOS, os.path.basename(d)))

print("\nTudo salvo em:", RESULTADOS)


data.yaml salvo.
Salvo: /content/drive/MyDrive/LIA/objects-phone-v3/train-5
Salvo: /content/drive/MyDrive/LIA/objects-phone-v3/val-9
Salvo: /content/drive/MyDrive/LIA/objects-phone-v3/predict_video-3

Tudo salvo em: /content/drive/MyDrive/LIA/objects-phone-v3
